# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
from jax.typing import ArrayLike
import jax.numpy as jnp
from functools import partial
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    worm_coherent_information
)
import time
import json
import os
import jax
n_cpus = os.cpu_count()
n_used_cpus = n_cpus
print("Number of CPUs available: {}".format(n_cpus))
print("Number of used CPUs: {}".format(n_used_cpus))
jax.config.update('jax_num_cpu_devices', n_used_cpus)
# jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


In [5]:
moebius_setup = {"length": 3, "width": 3, "p": 3}

worm_setup = {}
worm_setup["num_samples"] = 20 * n_cpus
worm_setup["num_worms"] = 200
worm_setup["burn_in_steps"] = 1000
worm_setup["max_worm_steps"] = 2000
plaquette_keys_setup = {}
plaquette_keys_setup["worm_master_seed"] = 144
plaquette_keys_setup["error_master_seed"] = 42

vertex_keys_setup = {}
vertex_keys_setup["worm_master_seed"] = 144
vertex_keys_setup["error_master_seed"] = 42
gamma_t = 0.08

In [6]:
moebius_code = MoebiusCodeTwoOddPrime(
    length=moebius_setup["length"], width=moebius_setup["width"], d=2 * moebius_setup["p"])
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * moebius_setup["p"], gamma_t=gamma_t
)
print(f"Single error probabilities: {em_lindblad.get_probabilities()}")

Single error probabilities: [8.5760629e-01 6.8389967e-02 2.7342141e-03 1.4567375e-04 2.7339906e-03
 6.8389818e-02]


In [7]:
coherent_information, plaquette_conditional_entropy, vertex_conditional_entropy = worm_coherent_information(
    gamma_t=gamma_t,
    moebius_setup=moebius_setup,
    worm_setup=worm_setup,
    plaquette_keys_setup=plaquette_keys_setup,
    vertex_keys_setup=vertex_keys_setup
)

print(f"Plaquette conditional entropy: {plaquette_conditional_entropy}")
print(f"Vertex conditional entropy: {vertex_conditional_entropy}")
print(f"Coherent Information: {coherent_information}")

Plaquette conditional entropy: 0.0
Vertex conditional entropy: 0.010022747330367565
Coherent Information: 0.989977240562439


In [ ]:
num_gamma = 20
gamma_min = 0.08
gamma_max = 0.4
gamma_t_list = (np.linspace(0.08, 0.3, num_gamma)).tolist()

result = {}
result["gamma_t"] = gamma_t_list

result["worm_setup"] = worm_setup
result["moebius_setup"] = moebius_setup

max_integer = 1_000_000
result["plaquette_worm_master_seed"] = (
    np.random.randint(0, max_integer, num_gamma)).tolist()
result["plaquette_error_master_seed"] = (
    np.random.randint(0, max_integer, num_gamma)).tolist()

result["vertex_worm_master_seed"] = (
    np.random.randint(0, max_integer, num_gamma)).tolist()
result["vertex_error_master_seed"] = (
    np.random.randint(0, max_integer, num_gamma)).tolist()

# This is to save on which machine the results are obtained
with open('/sys/devices/virtual/dmi/id/product_name') as f:
    result["machine_id"] = f.read()

with open('/sys/devices/virtual/dmi/id/sys_vendor') as f:
    result["vendor"] = f.read()

result["number_of_available_cpus"] = n_cpus
result["number_of_used_cpus"] = n_cpus

result["plaquette_conditional_entropy"] = []
result["vertex_conditional_entropy"] = []
result["coherent_information"] = []

In [ ]:
start = time.time()
for index in range(num_gamma):
    plaquette_keys_setup = {}
    plaquette_keys_setup["worm_master_seed"] = result["plaquette_worm_master_seed"][index]
    plaquette_keys_setup["error_master_seed"] = result["plaquette_error_master_seed"][index]

    vertex_keys_setup = {}
    vertex_keys_setup["worm_master_seed"] = result["plaquette_worm_master_seed"][index]
    vertex_keys_setup["error_master_seed"] = result["plaquette_error_master_seed"][index]

    coherent_information, plaquette_ce, vertex_ce = worm_coherent_information(
        gamma_t=result["gamma_t"][index],
        moebius_setup=moebius_setup,
        worm_setup=worm_setup,
        plaquette_keys_setup=plaquette_keys_setup,
        vertex_keys_setup=vertex_keys_setup
    )

    print(f"Gamma: {result["gamma_t"][index]}")
    print(f"Coherent Information: {coherent_information}")

    result["plaquette_conditional_entropy"].append(plaquette_ce.tolist())
    result["vertex_conditional_entropy"].append(vertex_ce.tolist())
    result["coherent_information"].append(coherent_information.tolist())
end = time.time()

computation_time = end - start

result["computation_time"] = computation_time
result["time_unit"] = "sec"

Gamma: 0.08
Coherent Information: 0.9937421679496765
Gamma: 0.3
Coherent Information: -0.14547276496887207


In [30]:
save = True

filename = ("coherent_information_moebius_length_" +
            str(moebius_setup["length"]) +
            "_width_" +
            str(moebius_setup["width"]) +
            "_p_" +
            str(moebius_setup["p"]) + ".json"
            )

if save:
    with open(filename, "w") as fp:
        json.dump(result, fp)